# **Import Required Library**

In [1]:
!pip install datasets
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import evaluate
import torch
import time
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.normalizers import NFKC
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast
from datasets import concatenate_datasets
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from huggingface_hub import notebook_login


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
ds = load_dataset("datablations/c4-filter-small", split="train")
ds
ds = ds.select_columns(["text"])
ds = ds.train_test_split(test_size=0.1)

README.md:   0%|          | 0.00/791 [00:00<?, ?B/s]

data/train-00000-of-00001-091e566583af27(…):   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

# **Initialized Tokenizer**

In [ ]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.normalizeer = NFKC()
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size= 30512,
    special_tokens = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
)
tokenizer.train_from_iterator(ds["train"]["text"], trainer)
tokenizer.save("gpt_tokenizer.json")



In [ ]:
def tokenize(ds, tokenizer):
  outputs = tokenizer.encode_batch(ds["text"])
  return {
      "input_ids": [enc.ids for enc in outputs]
  }
tokenized_ds = ds.map(tokenize,fn_kwargs={"tokenizer": tokenizer},
                      remove_columns=["text"], batched=True, num_proc=20) #input_ids, attention_mask

Map (num_proc=20):   0%|          | 0/90000 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
tokenizer = PreTrainedTokenizerFast(tokenizer_file="gpt_tokenizer.json")
tokenizer.add_special_tokens({
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "mask_token": "<mask>"
})
tokenizer.save_pretrained("gpt-tokenizer")

('gpt-tokenizer/tokenizer_config.json', 'gpt-tokenizer/tokenizer.json')

# **Load Trained Tokenizer**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("shomin/gpt2-small-c4")


config.json:   0%|          | 0.00/834 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(ds, tokenizer):
  outputs = tokenizer(ds["text"])
  return {
      "input_ids": outputs["input_ids"]
  }
tokenized_ds = ds.map(tokenize,fn_kwargs={"tokenizer": tokenizer},
                      remove_columns=["text"], batched=True, num_proc=20) #input_ids, attention_mask

Map (num_proc=20):   0%|          | 0/90000 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#Dict[input_ids, labels]
block_size = 256
def group_text(tokenized_ds, block_size):
  concatenated = {k: sum(tokenized_ds[k], []) for k in tokenized_ds.keys()}
  total_length = len(concatenated["input_ids"])
  if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
  results = {
        k: [concatenated[k][i : i + block_size] for i in range(0, total_length, block_size)]
        for k in concatenated.keys()
  }
  results["labels"] = results["input_ids"].copy()
  return results

gpt_ds = tokenized_ds.map(group_text,fn_kwargs={'block_size': block_size}, batched=True, num_proc=20)



Map (num_proc=20):   0%|          | 0/90000 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/10000 [00:00<?, ? examples/s]

# **Train From Scratch**

In [ ]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=256,
    n_ctx=256,
    n_embd=256,
    n_layer=2,
    n_head=4,
    attn_pdrop=0.2,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)
model = GPT2LMHeadModel(config)

training_args = TrainingArguments(
    output_dir="gpt-small-c4",
    logging_dir="logs",

    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    fp16=True
)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gpt_ds["train"],
    eval_dataset=gpt_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)
trainer.train()
trainer.push_to_hub("Final clean model")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Step,Training Loss,Validation Loss
100,9.459467,8.822516
200,8.409371,7.999514
300,7.774028,7.585864
400,7.525453,7.442852
500,7.411205,7.336589
600,7.302393,7.235301
700,7.203911,7.140017
800,7.118732,7.058455
900,7.044442,6.988776
1000,6.989458,6.928644


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mall-c4/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...mall-c4/model.safetensors: 100%|##########| 37.8MB / 37.8MB            

CommitInfo(commit_url='https://huggingface.co/shomin/gpt-small-c4/commit/ee72540849f111de12ac6f367a0d9d6932e2c9d2', commit_message='Final clean model', commit_description='', oid='ee72540849f111de12ac6f367a0d9d6932e2c9d2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shomin/gpt-small-c4', endpoint='https://huggingface.co', repo_type='model', repo_id='shomin/gpt-small-c4'), pr_revision=None, pr_num=None)

# **Continuous Training**

In [ ]:
model = GPT2LMHeadModel.from_pretrained("shomin/gpt-small-c4")

training_args = TrainingArguments(
    output_dir="gpt-small-c4",
    logging_dir="logs",

    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    fp16=True
)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gpt_ds["train"],
    eval_dataset=gpt_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)
trainer.train()
trainer.push_to_hub("Final clean model")

config.json:   0%|          | 0.00/834 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mall-c4/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...mall-c4/model.safetensors: 100%|##########| 37.8MB / 37.8MB            

CommitInfo(commit_url='https://huggingface.co/shomin/gpt-small-c4/commit/661ae6b68c22a174643cc9f0e27e5953e87218cb', commit_message='Final clean model', commit_description='', oid='661ae6b68c22a174643cc9f0e27e5953e87218cb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shomin/gpt-small-c4', endpoint='https://huggingface.co', repo_type='model', repo_id='shomin/gpt-small-c4'), pr_revision=None, pr_num=None)

# **Fine-tuning**

In [3]:
dataset = load_dataset("imdb")
full_ds = concatenate_datasets([dataset["train"], dataset["test"]])
pos_ds = full_ds.filter(lambda x: x["label"] == 1)
neg_ds = full_ds.filter(lambda x: x["label"] == 0)
pos_split = pos_ds.train_test_split(test_size=0.2, seed=42)
neg_split = neg_ds.train_test_split(test_size=0.2, seed=42)
train_dataset = concatenate_datasets([pos_split["train"], neg_split["train"]]).shuffle(seed=42)
test_dataset = concatenate_datasets([pos_split["test"], neg_split["test"]]).shuffle(seed=42)
dataset["train"] = train_dataset
dataset["test"] = test_dataset
print(f"Train size: {len(dataset['train'])} (Pos: {len(pos_split['train'])}, Neg: {len(neg_split['train'])})")
print(f"Test size: {len(dataset['test'])} (Pos: {len(pos_split['test'])}, Neg: {len(neg_split['test'])})")


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 40000 (Pos: 20000, Neg: 20000)
Test size: 10000 (Pos: 5000, Neg: 5000)


In [4]:
model_name = "shomin/gpt-small-c4"
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/834 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/246 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def tokenize(ds):
  return tokenizer(
      ds["text"], padding="max_length", truncation=True, max_length=256
  )
tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format("torch")

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
id2label = {0: 'Negative', 1: 'Positive'}
label2id = {'Negative': 0, 'Positive': 1}
model = AutoModelForSequenceClassification.from_pretrained(
    "shomin/gpt-small-c4",
    num_labels=2, id2label=id2label, label2id=label2id
)
model.config.pad_token_id = tokenizer.pad_token_id



model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: shomin/gpt-small-c4
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = logits.argmax(axis=-1)
  return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="imdb-gpt-small",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    learning_rate= 5e-6,
    fp16=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.260257,0.339980,0.861700
2,0.241454,0.359471,0.859000
3,0.226989,0.354939,0.862900
4,0.219105,0.352203,0.863400
5,0.207877,0.377794,0.860200
6,0.207850,0.367474,0.860200
7,0.201220,0.374035,0.860000
8,0.193399,0.375948,0.859700
9,0.191542,0.379482,0.859800
10,0.189114,0.380863,0.859600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.21388085083007813, metrics={'train_runtime': 522.1547, 'train_samples_per_second': 766.057, 'train_steps_per_second': 11.97, 'total_flos': 971086233600000.0, 'train_loss': 0.21388085083007813, 'epoch': 10.0})

# **Inference**

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "shomin/gpt-imdb"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id
idx_2_label = {0: 'Negative', 1: 'Positive'}
def predict_sentiment(text):
    model.eval()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        conf, pred_idx = torch.max(probs, dim=-1)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    inference_time = (end_time - start_time) * 1000

    return idx_2_label[pred_idx.item()], conf.item(), inference_time
reviews = [
    'I expected so much more. The pacing was slow and the dialogue felt unnatural.',
    'This is the kind of film that stays with you long after the credits roll.',
    'Terribly executed from start to finish. I wouldn’t recommend it to anyone.'
]

total_time = 0
print(f"--- GPT-2 INFERENCE PERFORMANCE ({device.type.upper()}) ---")

for r in reviews:
    label, conf, duration = predict_sentiment(r)
    total_time += duration
    print(f"Review: {r[:60]}...")
    print(f" -> {label} ({conf:.2%}) | Time: {duration:.2f} ms\n")

avg_latency = total_time / len(reviews)
print(f"--- SUMMARY ---")
print(f"Average Latency: {avg_latency:.2f} ms per review")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

--- GPT-2 INFERENCE PERFORMANCE (CUDA) ---
Review: I expected so much more. The pacing was slow and the dialogu...
 -> Negative (99.02%) | Time: 5.81 ms

Review: This is the kind of film that stays with you long after the ...
 -> Positive (95.77%) | Time: 5.32 ms

Review: Terribly executed from start to finish. I wouldn’t recommend...
 -> Negative (98.65%) | Time: 5.68 ms

--- SUMMARY ---
Average Latency: 5.61 ms per review
